# CVE & CWE Mapping dataset (merged)

This notebook merges the two data sources used by this project into a single dataset:

1. The original CVE-to-CWE mapping dataset from
   [zefang-liu/cve-and-cwe-mapping-dataset](https://huggingface.co/datasets/zefang-liu/cve-and-cwe-mapping-dataset),
   enriched with CWE descriptions and parent-category information from the scraped CWE-699 software hierarchy.
2. The NVD CVE records scraped by this project (`scrapers/scraped_data/CVE/`), resolved against the
   CWE-699 software hierarchy.

The two sources are joined on the CVE identifier. Variables that exist in both sources are matched onto a
single shared column, while variables that exist in only one source are kept as separate columns and left
empty (`None`) for rows coming from the other source.

### Requirements

`pandas` for reading and combining the data, `huggingface_hub` for the upload, and `python-dotenv` to load
the `HF_TOKEN` from the project `.env` file.

In [18]:
!pip install pandas
!pip install -q huggingface_hub python-dotenv

## Configuration

In [19]:
import glob
import json
import os
import re
from pathlib import Path

import pandas as pd
from utils import remove_capital_words, remove_parentheses

# == KAGGLE/COLAB ==
# !git clone https://github.com/regularpooria/CWE_CVE_Mapping
# BASE_DIR = "./CWE_CVE_Mapping"

# == LOCAL ==
BASE_DIR = "../"

# Source 1: the original CVE -> CWE mapping dataset hosted on the Hub.
GLOBAL_DATASET_URL = "hf://datasets/zefang-liu/cve-and-cwe-mapping-dataset/Global_Dataset.csv"

# Folder the merged dataset is staged in before it is uploaded.
UPLOAD_DIR = Path("merged_hf_upload")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

## Load the original CVE -> CWE mapping dataset

In [20]:
df = pd.read_csv(GLOBAL_DATASET_URL)
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID
0,1,CVE-1999-0001,NaN,5.0,MEDIUM,ip_input.c in BSD-derived TCP/IP implementatio...,CWE-20
1,2,CVE-1999-0002,NaN,10.0,HIGH,Buffer overflow in NFS mountd gives root acces...,CWE-119
2,3,CVE-1999-0003,NaN,10.0,HIGH,Execute commands as root via buffer overflow i...,NVD-CWE-Other
3,4,CVE-1999-0004,NaN,5.0,MEDIUM,"MIME buffer overflow in email clients, e.g. So...",NVD-CWE-Other
4,5,CVE-1999-0005,NaN,10.0,HIGH,Arbitrary command execution via IMAP buffer ov...,NVD-CWE-Other


## Load the scraped CWE-699 software hierarchy

`CWE_SOFTWARE.json` stores the CWE-699 Software hierarchy. Each category has a `children` object whose keys
are the child CWE ids that belong to it.

In [21]:
with open(f"{BASE_DIR}/scrapers/scraped_data/CWE_SOFTWARE.json", "r", encoding="utf-8") as f:
    CWE = json.load(f)

list_of_CWEs = set()
for category_id, category in CWE.items():
    list_of_CWEs.add(category_id)
    list_of_CWEs |= set(list(category["children"].keys()))

## Load every scraped NVD CVE file

All scraped NVD CSVs live under `scrapers/scraped_data/CVE/`. They are discovered with a glob instead of
hard-coded file names, so a new yearly file (for example `nvd_2027_cwe699.csv`) is picked up automatically
the next time this notebook runs.

In [22]:
CVE_DIR = Path(BASE_DIR) / "scrapers" / "scraped_data" / "CVE"
CVE_files = sorted(glob.glob(str(CVE_DIR / "*.csv")))

print(f"Found {len(CVE_files)} CVE files:")
for file in CVE_files:
    print(f"  - {Path(file).name}")

CVE = pd.DataFrame()
for i, file in enumerate(CVE_files):
    if i == 0:
        CVE = pd.read_csv(file)
        continue
    CVE = pd.concat([CVE, pd.read_csv(file)], ignore_index=True)

print(f"\nCombined {len(CVE):,} rows x {len(CVE.columns)} columns")

Found 4 CVE files:
  - nvd_2023_cwe699.csv
  - nvd_2024_cwe699.csv
  - nvd_2025_cwe699.csv
  - nvd_2026_cwe699.csv

Combined 88,146 rows x 9 columns


### Inspect the combined NVD data

In [23]:
print(CVE["published_year"].value_counts().sort_index())
CVE.head(5)

published_year
2023    18282
2024    24906
2025    26816
2026    18142
Name: count, dtype: int64


,cve_id,published,last_modified,published_year,description,all_nvd_cwes,cwe_699_matches,number_of_nvd_cwes,number_of_cwe_699_matches
0,CVE-2023-0028,2023-01-01T01:15:12.627,2026-06-17T05:24:37.547,2023,Cross-site Scripting (XSS) - Stored in GitHub ...,CWE-79,CWE-79,1,1
1,CVE-2023-22451,2023-01-02T16:15:11.063,2026-06-17T05:35:28.713,2023,Kiwi TCMS is an open source test management sy...,CWE-521,CWE-521,1,1
2,CVE-2023-0038,2023-01-03T14:15:10.390,2026-06-17T05:24:38.627,2023,"The ""Survey Maker – Best WordPress Survey Plug...",CWE-79,CWE-79,1,1
3,CVE-2023-22456,2023-01-03T19:15:10.483,2026-06-17T05:35:29.547,2023,"ViewVC, a browser interface for CVS and Subver...",CWE-79,CWE-79,1,1
4,CVE-2023-0046,2023-01-04T12:15:08.613,2026-06-17T05:24:39.563,2023,Improper Restriction of Names for Files and Ot...,CWE-641,CWE-641,1,1


## Clean the original dataset

Remove ambiguous CWE mappings, keep only CWE-699 software weaknesses, extract the year, and attach the
parent-category and CWE-description information from the scraped hierarchy.

In [24]:
# Removing any ambiguous cases.
values_to_remove = ["NVD-CWE-Other", "NVD-CWE-noinfo"]
df = df[~df["CWE-ID"].isin(values_to_remove)]
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID
0,1,CVE-1999-0001,NaN,5.0,MEDIUM,ip_input.c in BSD-derived TCP/IP implementatio...,CWE-20
1,2,CVE-1999-0002,NaN,10.0,HIGH,Buffer overflow in NFS mountd gives root acces...,CWE-119
6,7,CVE-1999-0007,NaN,5.0,MEDIUM,Information from SSL-encrypted sessions via PK...,CWE-327
25,26,CVE-1999-0027,NaN,7.2,HIGH,root privileges via buffer overflow in eject c...,CWE-119
176,177,CVE-1999-0179,NaN,5.0,MEDIUM,Windows NT crashes or locks up when a Samba cl...,CWE-17


In [25]:
# Extract the year
def extract_year(CVE_ID):
    return int(CVE_ID.split("-")[1])

df["YEAR"] = df["CVE-ID"].apply(extract_year)
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID,YEAR
0,1,CVE-1999-0001,NaN,5.0,MEDIUM,ip_input.c in BSD-derived TCP/IP implementatio...,CWE-20,1999
1,2,CVE-1999-0002,NaN,10.0,HIGH,Buffer overflow in NFS mountd gives root acces...,CWE-119,1999
6,7,CVE-1999-0007,NaN,5.0,MEDIUM,Information from SSL-encrypted sessions via PK...,CWE-327,1999
25,26,CVE-1999-0027,NaN,7.2,HIGH,root privileges via buffer overflow in eject c...,CWE-119,1999
176,177,CVE-1999-0179,NaN,5.0,MEDIUM,Windows NT crashes or locks up when a Samba cl...,CWE-17,1999


In [26]:
# Check to see if they are in CWE699 (SOFTWARE)
def is_in_CWE699(CWE_ID):
    id = CWE_ID.split("-")[-1]
    return id in list_of_CWEs

df["IS-IN-CWE699"] = df["CWE-ID"].apply(is_in_CWE699)
df = df[~df["IS-IN-CWE699"] == False]
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True
222,223,CVE-1999-0226,NaN,10.0,HIGH,Windows NT TCP/IP processes fragmented IP pack...,CWE-19,1999,True
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True
382,383,CVE-1999-0387,NaN,7.8,HIGH,A legacy credential caching mechanism used in ...,CWE-255,1999,True


In [27]:
df = df.rename(columns={"DESCRIPTION": "CVE-DESCRIPTION"})
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,CVE-DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True
222,223,CVE-1999-0226,NaN,10.0,HIGH,Windows NT TCP/IP processes fragmented IP pack...,CWE-19,1999,True
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True
382,383,CVE-1999-0387,NaN,7.8,HIGH,A legacy credential caching mechanism used in ...,CWE-255,1999,True


In [28]:
# Children to parent mapping

children_to_parent = {}
for category_id, category in CWE.items():
    for child_id, child in category["children"].items():
        child_id = str(child_id)
        item_to_add = str(category_id)
        item_to_add = (f"CWE-{category_id}", category["summary"])
        if child_id in children_to_parent:
            children_to_parent[child_id]["parents"].append(item_to_add)
        else:
            children_to_parent[child_id] = child
            children_to_parent[child_id]["parents"] = [item_to_add]

In [29]:
def get_parent_id(CWE_ID):
    id = CWE_ID.split("-")[-1]  # "CWE-199" -> "199"
    if id not in children_to_parent:
        return pd.NA

    return children_to_parent[id]["parents"]

df["PARENTS"] = df["CWE-ID"].apply(get_parent_id)
df = df[~df["PARENTS"].isna()]
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,CVE-DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699,PARENTS
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True,"[(CWE-389, This category includes weaknesses t..."
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re..."
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re..."
485,486,CVE-1999-0491,NaN,4.6,MEDIUM,The prompt parsing in bash allows a local user...,CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel..."
502,503,CVE-1999-0509,NaN,10.0,HIGH,"Perl, sh, csh, or other shell interpreters are...",CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel..."


In [30]:
def get_CWE_description(CWE_ID):
    id = CWE_ID.split("-")[-1]  # "CWE-199" -> "199"
    return children_to_parent[id]["description"]

df["CWE-DESCRIPTION"] = df["CWE-ID"].apply(get_CWE_description)
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,CVE-DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699,PARENTS,CWE-DESCRIPTION
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True,"[(CWE-389, This category includes weaknesses t...",The product does not check the return value fr...
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re...",The product copies an input buffer to an outpu...
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re...",The product copies an input buffer to an outpu...
485,486,CVE-1999-0491,NaN,4.6,MEDIUM,The prompt parsing in bash allows a local user...,CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel...",The product constructs all or part of a code s...
502,503,CVE-1999-0509,NaN,10.0,HIGH,"Perl, sh, csh, or other shell interpreters are...",CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel...",The product constructs all or part of a code s...


In [31]:
# Normalize the CVE description (drop product names, versions and parentheses).
df["CVE-DESCRIPTION-NORMALIZED"] = df["CVE-DESCRIPTION"].apply(
    lambda x: remove_capital_words(remove_parentheses(x))
)
df.head(5)

,ID,CVE-ID,CVSS-V3,CVSS-V2,SEVERITY,CVE-DESCRIPTION,CWE-ID,YEAR,IS-IN-CWE699,PARENTS,CWE-DESCRIPTION,CVE-DESCRIPTION-NORMALIZED
195,196,CVE-1999-0199,9.8,7.5,CRITICAL,manual/search.texi in the GNU C Library (aka g...,CWE-252,1999,True,"[(CWE-389, This category includes weaknesses t...",The product does not check the return value fr...,manual/search.texi in the GNU C Library before...
279,280,CVE-1999-0284,NaN,7.5,HIGH,Denial of service to NT mail servers including...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re...",The product copies an input buffer to an outpu...,Denial of service to NT mail servers including...
380,381,CVE-1999-0385,NaN,10.0,HIGH,The LDAP bind function in Exchange 5.5 has a b...,CWE-120,1999,True,"[(CWE-1218, Weaknesses in this category are re...",The product copies an input buffer to an outpu...,The LDAP bind function in has a buffer overflo...
485,486,CVE-1999-0491,NaN,4.6,MEDIUM,The prompt parsing in bash allows a local user...,CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel...",The product constructs all or part of a code s...,The prompt parsing in bash allows a local user...
502,503,CVE-1999-0509,NaN,10.0,HIGH,"Perl, sh, csh, or other shell interpreters are...",CWE-94,1999,True,"[(CWE-137, Weaknesses in this category are rel...",The product constructs all or part of a code s...,"Perl, sh, csh, or other shell interpreters are..."


## Resolve CWE-699 children and categories for the NVD data

- Build `child_to_cats`, mapping every child CWE id to the category (or categories) that contain it.
- Validate that no CVE has an empty `description`.
- Derive `true_children`: the `cwe_699_matches` of each CVE filtered down to ids that are known CWE-699 children.
- Validate that at least one known child remains for every CVE.
- Derive `true_categories`: the sorted set of category ids that those children belong to.
- Normalize the CVE description.
- Attach `CWE-DESCRIPTION` and `IS-IN-CWE699`, the two fields the original dataset already has, using the
  same scraped hierarchy (`children_to_parent` / `list_of_CWEs`). Every NVD row is a CWE-699 match by
  construction, so `IS-IN-CWE699` is `True`; `CWE-DESCRIPTION` is looked up per CWE id.

In [32]:
def parse_ids(value):
    return re.findall(r"\d+", str(value))


child_to_cats = {}
for cat_id, category in CWE.items():
    for child_id in category["children"].keys():
        child_to_cats.setdefault(child_id, []).append(cat_id)

if CVE["description"].isna().any():
    raise ValueError("Some descriptions in CVE are empty, please check and remove")

CVE["true_children"] = CVE["cwe_699_matches"].apply(
    lambda v: [c for c in parse_ids(v) if c in child_to_cats]
)
if (CVE["true_children"].str.len() == 0).any():
    raise ValueError("Some CVEs have a cwe_699_matches value that isn't a known CWE-699 child")

CVE["true_categories"] = CVE["true_children"].apply(
    lambda cs: sorted({cat for c in cs for cat in child_to_cats[c]})
)

CVE["description_normalized"] = CVE["description"].apply(
    lambda x: remove_capital_words(remove_parentheses(x))
)

# Attach the CWE fields the original dataset already carries. `children_to_parent`
# was built above from the same scraped hierarchy, so the descriptions line up
# with the ones used for the original dataset's rows.
def get_cwe_descriptions(CWE_ID):
    descriptions = [
        children_to_parent[id]["description"]
        for id in parse_ids(CWE_ID)
        if id in children_to_parent
    ]
    return "|".join(descriptions) if descriptions else pd.NA


CVE["CWE-DESCRIPTION"] = CVE["cwe_699_matches"].apply(get_cwe_descriptions)
CVE["IS-IN-CWE699"] = CVE["cwe_699_matches"].apply(
    lambda v: all(id in list_of_CWEs for id in parse_ids(v))
)
CVE[["cve_id", "cwe_699_matches", "CWE-DESCRIPTION", "IS-IN-CWE699"]].head(5)

,cve_id,cwe_699_matches,CWE-DESCRIPTION,IS-IN-CWE699
0,CVE-2023-0028,CWE-79,The product does not neutralize or incorrectly...,True
1,CVE-2023-22451,CWE-521,The product does not require that users should...,True
2,CVE-2023-0038,CWE-79,The product does not neutralize or incorrectly...,True
3,CVE-2023-22456,CWE-79,The product does not neutralize or incorrectly...,True
4,CVE-2023-0046,CWE-641,The product constructs the name of a file or o...,True


## Merge the two sources into one dataset

Match the variables that both sources share onto a single column name, keep the source-specific variables as
separate columns, and join both sources on `CVE-ID`.

Shared variables:

| Original dataset | NVD dataset | Merged column |
| --- | --- | --- |
| `CVE-ID` | `cve_id` | `CVE-ID` |
| `YEAR` | `published_year` | `YEAR` |
| `CVE-DESCRIPTION` | `description` | `CVE-DESCRIPTION` |
| `CVE-DESCRIPTION-NORMALIZED` | `description_normalized` | `CVE-DESCRIPTION-NORMALIZED` |
| `CWE-ID` | `cwe_699_matches` | `CWE-ID` |
| `CWE-DESCRIPTION` | `CWE-DESCRIPTION` | `CWE-DESCRIPTION` |
| `IS-IN-CWE699` | `IS-IN-CWE699` | `IS-IN-CWE699` |

Everything else is kept as its own column and filled with `None` for rows that come from the other source.

In [33]:
# Match the variables that both sources share onto one column name.
GLOBAL_RENAME = {
    "CVE-ID": "CVE-ID",
    "YEAR": "YEAR",
    "CVE-DESCRIPTION": "CVE-DESCRIPTION",
    "CVE-DESCRIPTION-NORMALIZED": "CVE-DESCRIPTION-NORMALIZED",
    "CWE-ID": "CWE-ID",
}
NVD_RENAME = {
    "cve_id": "CVE-ID",
    "published_year": "YEAR",
    "description": "CVE-DESCRIPTION",
    "description_normalized": "CVE-DESCRIPTION-NORMALIZED",
    "cwe_699_matches": "CWE-ID",
}

df = df.rename(columns=GLOBAL_RENAME)
CVE = CVE.rename(columns=NVD_RENAME)

# The original dataset can hold several rows per CVE (one per CWE); collapse it
# to a single row per CVE so it can be joined on CVE-ID.
MULTI_VALUE_COLUMNS = ["CWE-ID", "CWE-DESCRIPTION", "PARENTS"]


def join_unique(values):
    unique = []
    for value in values.dropna():
        if value not in unique:
            unique.append(value)
    return "|".join(str(v) for v in unique) if unique else pd.NA


df = df.groupby("CVE-ID", as_index=False).agg(
    {c: (join_unique if c in MULTI_VALUE_COLUMNS else "first") for c in df.columns if c != "CVE-ID"}
)

# Outer join on CVE-ID so every CVE from either source is kept.
MATCHED_COLUMNS = ["YEAR", "CVE-DESCRIPTION", "CVE-DESCRIPTION-NORMALIZED", "CWE-ID", "CWE-DESCRIPTION", "IS-IN-CWE699"]

df = df.add_suffix("__global").rename(columns={"CVE-ID__global": "CVE-ID"})
CVE = CVE.add_suffix("__nvd").rename(columns={"CVE-ID__nvd": "CVE-ID"})

merged = pd.merge(df, CVE, on="CVE-ID", how="outer")

# Coalesce the shared variables and drop the source-specific duplicates.
for column in MATCHED_COLUMNS:
    merged[column] = merged[f"{column}__global"].combine_first(merged[f"{column}__nvd"])
    merged = merged.drop(columns=[f"{column}__global", f"{column}__nvd"])

# Source-specific columns only exist on one side, so strip their suffix.
merged.columns = [
    c[:-8] if c.endswith("__global") else c[:-5] if c.endswith("__nvd") else c
    for c in merged.columns
]

COLUMN_ORDER = [
    "CVE-ID", "YEAR", "CVE-DESCRIPTION", "CVE-DESCRIPTION-NORMALIZED", "CWE-ID",
    "CWE-DESCRIPTION", "PARENTS", "IS-IN-CWE699", "CVSS-V3", "CVSS-V2", "SEVERITY",
    "ID", "published", "last_modified", "all_nvd_cwes", "number_of_nvd_cwes",
    "number_of_cwe_699_matches", "true_children", "true_categories",
]
merged = merged[COLUMN_ORDER]

for column in ["YEAR", "ID", "number_of_nvd_cwes", "number_of_cwe_699_matches"]:
    merged[column] = merged[column].astype("Int64")

print(f"Merged dataset: {len(merged):,} rows x {len(merged.columns)} columns")
merged.tail(5)

Merged dataset: 148,238 rows x 19 columns


,CVE-ID,YEAR,CVE-DESCRIPTION,CVE-DESCRIPTION-NORMALIZED,CWE-ID,CWE-DESCRIPTION,PARENTS,IS-IN-CWE699,CVSS-V3,CVSS-V2,SEVERITY,ID,published,last_modified,all_nvd_cwes,number_of_nvd_cwes,number_of_cwe_699_matches,true_children,true_categories
148233,CVE-2026-9976,2026,Inappropriate implementation in USB in Google ...,Inappropriate implementation in USB in prior t...,CWE-94,The product constructs all or part of a code s...,NaN,True,NaN,NaN,NaN,<NA>,2026-05-28T23:16:55.833,2026-06-17T11:05:56.480,CWE-94,1,1,[94],[137]
148234,CVE-2026-9983,2026,Type Confusion in Skia in Google Chrome prior ...,Type Confusion in in prior to allowed a remote...,CWE-843,The product allocates or initializes a resourc...,NaN,True,NaN,NaN,NaN,<NA>,2026-05-28T23:16:56.537,2026-06-17T11:05:58.417,CWE-843,1,1,[843],[136]
148235,CVE-2026-9989,2026,Inappropriate implementation in Media in Googl...,Inappropriate implementation in in prior to al...,CWE-346,The product does not properly verify that the ...,NaN,True,NaN,NaN,NaN,<NA>,2026-05-28T23:16:57.140,2026-06-17T11:05:59.290,CWE-346,1,1,[346],"[1214, 417]"
148236,CVE-2026-9996,2026,Out of bounds read in WebRTC in Google Chrome ...,Out of bounds read in in on prior to allowed a...,CWE-125,"The product reads data past the end, or before...",NaN,True,NaN,NaN,NaN,<NA>,2026-05-28T23:16:57.830,2026-06-17T11:06:00.293,CWE-125,1,1,[125],[1218]
148237,CVE-2026-9998,2026,Integer overflow in Skia in Google Chrome prio...,Integer overflow in in prior to allowed a remo...,CWE-472,The web application does not sufficiently veri...,NaN,True,NaN,NaN,NaN,<NA>,2026-05-28T23:16:58.037,2026-06-17T11:06:00.577,CWE-472,1,1,[472],[19]


## Write the dataset card

Hugging Face renders the `README.md` at the root of a dataset repository as its landing page. It is written
into the staging folder so it is committed together with the CSV.

In [34]:
DATASET_CARD = '''---
language:
- en
tags:
- cybersecurity
- vulnerability
- cve
- nvd
- cwe
- software-security
task_categories:
- text-classification
pretty_name: CVE-CWE Software Weakness Mapping Dataset
license: apache-2.0
---

# CVE-CWE Software Weakness Mapping Dataset

## Dataset description

This dataset merges two CVE-to-CWE data sources into a single table:

- The original CVE-to-CWE mapping dataset from
  [zefang-liu/cve-and-cwe-mapping-dataset](https://huggingface.co/datasets/zefang-liu/cve-and-cwe-mapping-dataset),
  enriched with CWE descriptions and parent-category information from the scraped CWE-699 software hierarchy.
- The NVD CVE records scraped by this project, resolved against the CWE-699 software hierarchy.

The two sources are joined on the CVE identifier. Variables shared by both sources are matched onto a single
column; variables that exist in only one source are kept as separate columns and left empty for rows coming
from the other source.

## Dataset structure

The dataset is provided as `CVE_CWE_Dataset.csv`.

Variables shared by both sources:

- `CVE-ID`: CVE identifier (join key).
- `YEAR`: year the CVE was published.
- `CVE-DESCRIPTION`: vulnerability description.
- `CVE-DESCRIPTION-NORMALIZED`: normalized vulnerability description.
- `CWE-ID`: associated CWE identifier(s), pipe separated.
- `CWE-DESCRIPTION`: description of the associated weakness (pipe separated for multiple CWEs).
- `IS-IN-CWE699`: whether the CWE belongs to the CWE-699 Software category.

Variables from the original (zefang-liu) dataset only:

- `PARENTS`: parent CWE category or categories.
- `CVSS-V3`: CVSS v3 score.
- `CVSS-V2`: CVSS v2 score.
- `SEVERITY`: severity rating.
- `ID`: identifier from the original dataset.

Variables from the NVD dataset only:

- `published`: publication timestamp.
- `last_modified`: last modification timestamp.
- `all_nvd_cwes`: all CWE identifiers NVD associates with the CVE (pipe separated).
- `number_of_nvd_cwes`: count of `all_nvd_cwes`.
- `number_of_cwe_699_matches`: count of `CWE-ID`.
- `true_children`: the CWE-699 child ids from `CWE-ID` that exist in `CWE_SOFTWARE.json` (list).
- `true_categories`: the CWE-699 category ids those children belong to (sorted list).

## Data sources

- CVE-to-CWE mappings and vulnerability descriptions are sourced from the `zefang-liu/cve-and-cwe-mapping-dataset`
  dataset on Hugging Face.
- Raw CVE records are collected from the NVD.
- CWE descriptions, hierarchy and CWE-699 Software category membership are collected from the CWE data scraped
  by this project.

The dataset is generated by `dataset/merge_datasets.ipynb`.

## Intended uses

This dataset is intended for research and experimentation involving:

- CVE-to-CWE classification.
- Vulnerability search and retrieval.
- Security text embedding and similarity experiments.
- Analysis of software weakness categories.

## Limitations

The mappings reflect the upstream NVD, CVE and CWE data and may contain incomplete, ambiguous, or changing
vulnerability information. The dataset should not be treated as a definitive security assessment or as a
replacement for reviewing the original NVD and CWE records.

## License and attribution

Please review and follow the licenses and attribution requirements of the upstream NVD, CVE and CWE sources
before redistributing or using this dataset.
'''

(UPLOAD_DIR / "README.md").write_text(DATASET_CARD, encoding="utf-8")
print(f"Wrote dataset card to {UPLOAD_DIR / 'README.md'}")

Wrote dataset card to merged_hf_upload/README.md


## Save and upload the merged dataset

Configuration:

- `HF_TOKEN` is read from the project `.env` file (loaded with `python-dotenv`). Create it at the repository
  root with `HF_TOKEN=hf_...`.
- The target repository defaults to `regularpooria/CVE_CWE_Dataset`. Override it by adding
  `HF_MERGED_DATASET_REPO_ID=your-user/your-repo` to `.env`.
- The upload creates the repository automatically if it does not exist yet (the token must have write access).

In [36]:
# Save the merged dataset locally and commit it to the Hugging Face dataset repository.
from dotenv import load_dotenv
from huggingface_hub import HfApi

load_dotenv()
HF_REPO_ID = os.environ.get("HF_MERGED_DATASET_REPO_ID", "regularpooria/CVE_CWE_Software_Mapping_Dataset")

DATASET_PATH = UPLOAD_DIR / "CVE_CWE_Dataset.csv"
merged.to_csv(DATASET_PATH, index=False)

api = HfApi(token=os.environ.get("HF_TOKEN"))
commit_url = api.upload_folder(
    folder_path=str(UPLOAD_DIR),
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message="Update merged CVE-CWE dataset",
)

print(f"Saved {len(merged):,} rows to {DATASET_PATH}")
print(f"Committed dataset and dataset card to {commit_url}")

Saved 148,238 rows to merged_hf_upload/CVE_CWE_Dataset.csv
Committed dataset and dataset card to https://huggingface.co/datasets/regularpooria/CVE_CWE_Software_Mapping_Dataset/commit/c5cb406753d293815de4cac633f81686abc04122
